# 03 · Process into the final dataset

Takes the split from notebook 02 and writes the final, licence-clean dataset.

1. Keep only photos whose licence allows this use: CC0, CC BY, CC BY-SA, CC BY-NC and CC BY-NC-SA.
2. Read every photo, check its SHA-1 against the manifest and compute its perceptual hash.
3. Drop a validation or test photo within 4 bits of a photo in an earlier split, or identical to one in its own split.
4. Make every species with at least 20 photos a class. Other species go to `unknown_pool`, which is never trained on.
5. Write `data/<split>/<species>/`, `metadata.csv`, `classes.csv`, `processing_report.json` and sample grids with credits.

- **Reads:** the raw dataset (01) and the split dataset (02).
- **Saves:** `<your username>/serpentguard-indian-snakes-clean`, private.
- **Time:** about 30 minutes on CPU.

Take `class_id` from `classes.csv` for every split: `val/` and `test/` lack some species folders, so loaders that
number classes by folder (plain `ImageFolder`) would mislabel photos.

In [1]:
import hashlib
import io
import json
import shutil
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
from PIL import Image
from scipy.fft import dctn

RAW_HANDLE = "puneetbajajiitm/serpentguard-indian-snakes"
SPLITS_HANDLE = "puneetbajajiitm/serpentguard-m2-splits"
DATASET_SLUG = "serpentguard-indian-snakes-clean"
UPLOAD = True
SAMPLE_ROWS = None  # a small number makes a quick trial run

ALLOWED_LICENCES = {"cc0", "cc-by", "cc-by-sa", "cc-by-nc", "cc-by-nc-sa"}
NEAR_COPY_BITS = 4
MIN_CLASS_PHOTOS = 20
TIER1_MIN_PHOTOS = 400
SPLIT_ORDER = ("train", "val", "test", "test_extra")
UNKNOWN = "unknown_pool"
ID_TYPES = {"obs_id": str, "photo_id": str, "observer_id": str, "sha1": str}

PHASH_SIZE, PHASH_KEEP = 32, 8
READ_WORKERS = 32
COMPARE_CHUNK = 1000
GRID_TILE, GRID_COLUMNS, WHO_PER_SPECIES, GRID_SEED = 112, 16, 4, 7

SCRATCH = Path("/kaggle/tmp/serpentguard-clean")
SCRATCH.mkdir(parents=True, exist_ok=True)
(SCRATCH / "figures").mkdir(exist_ok=True)
WORKING = Path("/kaggle/working")
POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

README = """# SerpentGuard Indian snakes, licence-clean (private, course use only)

IIT Madras BS, DS and AI Lab, Group 5. iNaturalist research-grade photos under CC0, CC BY, CC BY-SA,
CC BY-NC and CC BY-NC-SA only. All-rights-reserved and no-derivatives photos are not in this dataset.

data/<split>/<species>/<file>.jpg   train, val, test, test_extra, unknown_pool
metadata.csv   one row per photo with labels, split, licence and credit
classes.csv    one row per class. Use its class_id for every split
processing_report.json   counts and leakage checks
figures/       sample grids, each with an attribution CSV

Copyright stays with each photographer; see the license and attribution columns.
"""

In [2]:
def phash(body):
    with Image.open(io.BytesIO(body)) as image:
        gray = np.asarray(image.convert("L").resize((PHASH_SIZE, PHASH_SIZE), Image.LANCZOS), dtype=np.float32)
    low = dctn(gray, norm="ortho")[:PHASH_KEEP, :PHASH_KEEP].flatten()
    return f"{int(''.join('1' if bit else '0' for bit in low > np.median(low[1:])), 2):016x}"


def read_photo(row):
    body = (raw_root / "images" / row.file).read_bytes()
    return hashlib.sha1(body).hexdigest() == row.sha1, phash(body)


def to_bits(hashes):
    return np.array([int(value, 16) for value in hashes], dtype=np.uint64)


def closest_bits(hashes, bank):
    """Hamming distance from each hash to its nearest neighbour in the bank."""
    closest = np.full(len(hashes), 64)
    for start in range(0, len(hashes) if len(bank) else 0, COMPARE_CHUNK):
        xor = hashes[start:start + COMPARE_CHUNK, None] ^ bank[None, :]
        closest[start:start + COMPARE_CHUNK] = POPCOUNT[xor.view(np.uint8).reshape(*xor.shape, 8)].sum(axis=2).min(axis=1)
    return closest


def find_near_copies(frame):
    """Index of validation and test photos that repeat an earlier split, or their own split exactly."""
    drop, bank = [], np.array([], dtype=np.uint64)
    for split in SPLIT_ORDER:
        part = frame[frame["split"] == split]
        if split != "train":
            repeat = part["phash"].duplicated().to_numpy() | (closest_bits(to_bits(part["phash"]), bank) <= NEAR_COPY_BITS)
            drop.extend(part.index[repeat])
            part = part[~repeat]
        bank = np.concatenate([bank, to_bits(part["phash"])])
    return drop


def write_grid(frame, species, per_species, name, archive):
    """A grid of training photos, with a CSV crediting every tile."""
    rng = np.random.default_rng(GRID_SEED)
    picks = []
    for one in species:
        pool = frame[(frame["species"] == one) & (frame["split"] == "train")]
        picks.append(pool.iloc[rng.choice(len(pool), size=min(per_species, len(pool)), replace=False)])
    picks = pd.concat(picks)
    columns = per_species if per_species > 1 else GRID_COLUMNS
    canvas = Image.new("RGB", (columns * GRID_TILE, -(-len(picks) // columns) * GRID_TILE), "white")
    for i, path in enumerate(picks["path"]):
        with Image.open(io.BytesIO(archive.read(f"{path}"))) as image:
            canvas.paste(image.convert("RGB").resize((GRID_TILE, GRID_TILE)), ((i % columns) * GRID_TILE, (i // columns) * GRID_TILE))
    canvas.save(SCRATCH / "figures" / f"{name}.jpg", quality=90)
    picks[["species", "license", "attribution", "url"]].to_csv(SCRATCH / "figures" / f"{name}_attribution.csv", index_label="row")

## Licences, hashes and near copies

In [3]:
raw_root = Path(kagglehub.dataset_download(RAW_HANDLE))
splits_root = Path(kagglehub.dataset_download(SPLITS_HANDLE))
everything = pd.read_csv(splits_root / "splits.csv", dtype=ID_TYPES)
if SAMPLE_ROWS:
    everything = everything.sample(SAMPLE_ROWS, random_state=GRID_SEED).reset_index(drop=True)

removed_by_licence = everything.loc[~everything["license"].isin(ALLOWED_LICENCES), "license"].value_counts().to_dict()
df = everything[everything["license"].isin(ALLOWED_LICENCES)].reset_index(drop=True)
print(f"{len(everything):,} photos in the split table; {len(df):,} have an allowed licence; removed {removed_by_licence}")

with ThreadPoolExecutor(READ_WORKERS) as pool:
    df["sha1_verified"], df["phash"] = zip(*pool.map(read_photo, df.itertuples()))
print("SHA-1 mismatches:", int((~df["sha1_verified"]).sum()))

copies = find_near_copies(df)
copies_by_split = df.loc[copies, "split"].value_counts().to_dict()
df = df.drop(index=copies).reset_index(drop=True)
print(f"Near copies removed: {len(copies):,} {copies_by_split}")

181,503 photos in the split table; 121,751 have an allowed licence; removed {'all-rights-reserved': 56739, 'cc-by-nc-nd': 2902, 'cc-by-nd': 111}
SHA-1 mismatches: 0
Near copies removed: 185 {'val': 105, 'test_extra': 55, 'test': 25}


## Classes, tiers and the unknown pool

In [4]:
labelled = df["split"] != UNKNOWN
counts = df[labelled].groupby("species").size()
small = counts[counts < MIN_CLASS_PHOTOS].index
move = labelled & df["species"].isin(small)
df.loc[move, "path"] = UNKNOWN + "/" + df.loc[move, "path"].str.split("/", n=1).str[1]
df.loc[move, "split"] = UNKNOWN
df.loc[df["split"] == UNKNOWN, "tier"] = UNKNOWN

labelled = df["split"] != UNKNOWN
totals = df[labelled].groupby("species")["species"].transform("size")
df.loc[labelled, "tier"] = np.where(totals >= TIER1_MIN_PHOTOS, "tier1", "tier2")

classes = df[labelled].pivot_table(index="species", columns="split", values="photo_id", aggfunc="count", fill_value=0)
classes = classes.reindex(columns=list(SPLIT_ORDER), fill_value=0)
labels = df[labelled].drop_duplicates("species").set_index("species")[["genus", "family", "who_category", "tier", "m1_core27", "taxon_id"]]
classes = classes.join(labels)
classes["total"] = classes[list(SPLIT_ORDER)].sum(axis=1)
classes = classes.sort_values("total", ascending=False)
classes.insert(0, "class_id", range(len(classes)))
print(f"{len(classes)} classes: {classes['tier'].value_counts().to_dict()}; {move.sum():,} photos of {len(small)} species moved to {UNKNOWN}")

184 classes: {'tier2': 128, 'tier1': 56}; 367 photos of 24 species moved to unknown_pool


## Write the dataset

In [5]:
assert df["license"].isin(ALLOWED_LICENCES).all()
with zipfile.ZipFile(SCRATCH / "data.zip", "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
    for row in df.itertuples():
        archive.write(raw_root / "images" / row.file, row.path)

df.to_csv(SCRATCH / "metadata.csv", index=False)
classes.to_csv(SCRATCH / "classes.csv")
(SCRATCH / "README.md").write_text(README)

labelled_rows = df[df["split"] != UNKNOWN]
block = labelled_rows["split"].replace({"test_extra": "test"})
leakage = {f"{key}_in_more_than_one_of_train_val_testfold": int(labelled_rows.assign(block=block).groupby(key)["block"].nunique().gt(1).sum())
           for key in ("observer_id", "obs_id", "sha1", "phash")}
report = {
    "input_rows": len(everything),
    "removed_by_licence": removed_by_licence,
    "near_copies_removed_by_split": copies_by_split,
    "near_copy_bits": NEAR_COPY_BITS,
    "species_moved_to_unknown_pool": sorted(small),
    "photos_moved_to_unknown_pool": int(move.sum()),
    "sha1_mismatches": int((~df["sha1_verified"]).sum()),
    "kept_by_split": df["split"].value_counts().to_dict(),
    "photos": len(df),
    "classes": len(classes),
    "classes_by_tier": classes["tier"].value_counts().to_dict(),
    "licences": df["license"].value_counts().to_dict(),
    "leakage": leakage,
    "bytes_zip": (SCRATCH / "data.zip").stat().st_size,
}
(SCRATCH / "processing_report.json").write_text(json.dumps(report, indent=1, default=str))
print(json.dumps(report, indent=1, default=str))

with zipfile.ZipFile(SCRATCH / "data.zip") as archive:
    write_grid(df, list(classes.index), 1, "grid_all_classes", archive)
    write_grid(df, list(classes[classes["who_category"].astype(str) != "not_listed"].index), WHO_PER_SPECIES, "grid_who_listed", archive)

for name in ("metadata.csv", "classes.csv", "processing_report.json"):
    shutil.copy(SCRATCH / name, WORKING / name)
shutil.copytree(SCRATCH / "figures", WORKING / "figures", dirs_exist_ok=True)

if UPLOAD:
    kagglehub.dataset_upload(f"{kagglehub.whoami()['username']}/{DATASET_SLUG}", str(SCRATCH), version_notes="Licence-clean dataset")

{
 "input_rows": 181503,
 "removed_by_licence": {
  "all-rights-reserved": 56739,
  "cc-by-nc-nd": 2902,
  "cc-by-nd": 111
 },
 "near_copies_removed_by_split": {
  "val": 105,
  "test_extra": 55,
  "test": 25
 },
 "near_copy_bits": 4,
 "species_moved_to_unknown_pool": [
  "Ahaetulla flavescens",
  "Boiga gocool",
  "Boiga quincunciata",
  "Boiga westermanni",
  "Bungarus lividus",
  "Bungarus sindanus",
  "Craspedocephalus peltopelor",
  "Hebius clerki",
  "Hydrophis caerulescens",
  "Hydrophis fasciatus",
  "Melanophidium bilineatum",
  "Melanophidium punctatum",
  "Oligodon juglandifer",
  "Oligodon venustus",
  "Platyceps ladacensis",
  "Sibynophis sagittarius",
  "Trimeresurus uetzi",
  "Trimerodytes yunnanensis",
  "Uropeltis jerdoni",
  "Uropeltis maculata",
  "Uropeltis myhendrae",
  "Uropeltis shorttii",
  "Xenochrophis cerasogaster",
  "Xylophis captaini"
 ],
 "photos_moved_to_unknown_pool": 367,
 "sha1_mismatches": 0,
 "kept_by_split": {
  "train": 85067,
  "val": 17151,
  "t

Uploading: 100%|██████████| 66.3M/66.3M [00:03<00:00, 21.4MB/s]

Upload successful: /kaggle/tmp/serpentguard-clean/metadata.csv (63MB)
Starting upload for file /kaggle/tmp/serpentguard-clean/README.md



Uploading: 100%|██████████| 685/685 [00:00<00:00, 1.80kB/s]

Upload successful: /kaggle/tmp/serpentguard-clean/README.md (685B)
Starting upload for file /kaggle/tmp/serpentguard-clean/classes.csv


...


Uploading: 100%|██████████| 29.9k/29.9k [00:00<00:00, 79.1kB/s]

Upload successful: /kaggle/tmp/serpentguard-clean/figures/grid_all_classes_attribution.csv (29KB)
Starting upload for file /kaggle/tmp/serpentguard-clean/figures/grid_all_classes.jpg



Uploading: 100%|██████████| 1.00M/1.00M [00:01<00:00, 976kB/s]

Upload successful: /kaggle/tmp/serpentguard-clean/figures/grid_all_classes.jpg (976KB)


Your dataset has been created.
Files are being processed...
See at: https://kaggle.com/datasets/puneetbajajiitm/serpentguard-indian-snakes-clean
